# Bronze → Silver: inspecoes

**Entrada:** `` `v-energy`.bronze.tb_inspecoes `` (texto bruto das 3 abas de `inspecoes.xlsx`, 1.389 linhas).
**Saída:** `` `v-energy`.silver.tb_inspecoes `` (Delta, tipada e padronizada).
**Grão:** uma linha por inspeção de campo (`id_inspecao` único).

**O que este notebook faz.** Cada passo corresponde a um achado de `docs/auditoria_dados/auditoria_inspecoes.md`:
1. **Converte `data`** pelo formato detectado em cada valor (`yyyy-MM-dd`, `dd/MM/yyyy`, `dd.MM.yyyy` e o americano `MM-dd-yyyy`), nunca pela aba: 107 linhas da aba `dez` são de agosto a novembro.
2. **Padroniza o texto:** `inspetor` sem os espaços nas pontas (40 códigos), `resultado` nos 2 valores do contrato e as 4 formas de ausência de `tipo_irregularidade` (vazio, `-`, `NULL`, `N/A`) como nulo.
3. **Converte `kwh_recuperado`** para `DECIMAL(18,2)`, aceitando vírgula ou ponto como separador decimal.
4. **Valida antes de gravar:** padrões do contrato, coerência entre resultado, tipo e kWh, e toda UC presente na Silver de cadastro. Se uma regra crítica falhar, o notebook para e a Silver atual não é tocada.
5. **Grava só com `confirmar_publicacao = true`.** Quando o Workflow do Databricks existir (#97), a tarefa vai passar `true`; até lá, quem roda à mão troca o widget. Depois de gravar, confere a tabela e, se ela não bater com o que foi validado, volta a Silver para a versão anterior.

Documentação: `docs/silver/silver_inspecoes.md`.

In [0]:
# ruff: noqa: F821
# (spark, dbutils e display são variáveis globais do notebook no Databricks)
catalog = 'v-energy'
bronze_schema = 'bronze'
silver_schema = 'silver'

bronze_table = f'`{catalog}`.{bronze_schema}.tb_inspecoes'
silver_table = f'`{catalog}`.{silver_schema}.tb_inspecoes'
cadastro_table = f'`{catalog}`.{silver_schema}.tb_unidades_consumidoras'

COLUNAS_BRONZE = [
    'id_inspecao',
    'id_uc',
    'data',
    'inspetor',
    'resultado',
    'tipo_irregularidade',
    'kwh_recuperado',
]
COLUNAS_LINHAGEM = ['sheet', 'source_file', 'ingestion_datetime']
CONFIRMADA = 'IRREGULARIDADE CONFIRMADA'
SEM_IRREGULARIDADE = 'SEM IRREGULARIDADE'
# Sinônimos da origem, já em maiúsculas (auditoria, seção 3.3)
RESULTADOS = {
    'IRREGULARIDADE CONFIRMADA': CONFIRMADA,
    'CONFIRMADA': CONFIRMADA,
    'SEM IRREGULARIDADE': SEM_IRREGULARIDADE,
    'NADA CONSTA': SEM_IRREGULARIDADE,
}
MARCADORES_AUSENCIA = ['', '-', 'NULL', 'N/A']  # formas de "sem tipo" na origem
# Tipos de irregularidade da base; um tipo novo gera aviso (o contrato não tem enum)
TIPOS_IRREGULARIDADE = [
    'BY-PASS',
    'DESVIO DIRETO',
    'LACRE VIOLADO',
    'LIGACAO CLANDESTINA',
    'MEDIDOR ADULTERADO',
]
# Formato de cada valor de data → padrão do Spark (auditoria, seção 3.1)
FORMATOS_DATA = [
    (r'^\d{4}-\d{2}-\d{2}$', 'yyyy-MM-dd'),
    (r'^\d{2}/\d{2}/\d{4}$', 'dd/MM/yyyy'),
    (r'^\d{2}\.\d{2}\.\d{4}$', 'dd.MM.yyyy'),
    (r'^\d{2}-\d{2}-\d{4}$', 'MM-dd-yyyy'),  # americano
]
# Padrões de contracts/tabelas/inspecoes_historicas.yaml
PADRAO_ID_INSPECAO = r'^IN\d{7}$'
PADRAO_ID_UC = r'^UC\d{6}$'
PADRAO_INSPETOR = r'^INSP-(00[1-9]|0[1-3][0-9]|040)$'
FUSO_ORIGEM = 'America/Recife'  # "hoje" para barrar data futura
DATA_MINIMA = '2022-07-01'  # início da base; antes disso é sentinela ou erro de digitação

# O Workflow (#97) vai passar confirmar_publicacao=true. Um widget já existente conserva o valor.
try:
    dbutils.widgets.get('confirmar_publicacao')
except Exception:
    dbutils.widgets.dropdown('confirmar_publicacao', 'false', ['false', 'true'])
valor = dbutils.widgets.get('confirmar_publicacao').strip().lower()
if valor not in {'false', 'true'}:
    raise ValueError('confirmar_publicacao precisa ser false ou true.')
CONFIRMAR_PUBLICACAO = valor == 'true'
print(f'Publicação habilitada: {CONFIRMAR_PUBLICACAO}')

# Hoje no fuso da origem, fixado uma vez para toda a execução
HOJE = spark.sql(f"SELECT to_date(convert_timezone('{FUSO_ORIGEM}', current_timestamp()))").first()[
    0
]

Publicação habilitada: True


## 1. Leitura da Bronze
Fixa a versão atual da Bronze (`VERSION AS OF`): a validação e a escrita leem exatamente os mesmos dados, mesmo que a Bronze seja recarregada durante a execução. Depois confere se as colunas esperadas existem.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

versao_bronze = spark.sql(f'DESCRIBE HISTORY {bronze_table} LIMIT 1').first()['version']
df_bronze = spark.sql(f'SELECT * FROM {bronze_table} VERSION AS OF {versao_bronze}')
faltando = set(COLUNAS_BRONZE + COLUNAS_LINHAGEM) - set(df_bronze.columns)
if faltando:
    raise RuntimeError(f'Colunas ausentes na Bronze: {faltando}')

linhas_bronze = df_bronze.count()
print(f'{bronze_table} (versão {versao_bronze}): {linhas_bronze} linhas')
display(df_bronze.groupBy('sheet').count().orderBy('sheet'))

`v-energy`.bronze.tb_inspecoes (versão 20): 1389 linhas


sheet,count
ago-set,511
dez,383
out-nov,495


## 2. Padronização e tipagem
- **Texto:** `id_inspecao`, `id_uc`, `inspetor`, `resultado` e `tipo_irregularidade` sem espaços nas pontas e em maiúsculas.
- **`resultado`:** os sinônimos viram `IRREGULARIDADE CONFIRMADA` ou `SEM IRREGULARIDADE`. Um valor fora do mapa fica como veio e é barrado na validação.
- **`tipo_irregularidade`:** as 4 formas de ausência viram nulo.
- **`data`:** o formato é detectado por regex em cada valor e convertido com `try_to_timestamp`; data inexistente vira nulo e é barrada. A aba (`sheet`) fica só como linhagem.
- **`kwh_recuperado`:** vírgula vira ponto, e o valor é convertido com `try_cast` para `DECIMAL(18,2)`, a escala do contrato.
- As duplicatas exatas (as mesmas colunas da Bronze, já padronizadas) são removidas. Fica a linha da carga mais antiga (`ingestion_datetime`, depois `source_file` e `sheet`), então a linhagem gravada não muda entre execuções. A Bronze atual não tem nenhuma, mas a regra protege as próximas cargas.

In [0]:
def texto(coluna):
    """Remove espaços nas pontas e converte para maiúsculas."""
    return F.upper(F.trim(F.col(coluna)))


def para_data(coluna):
    """Texto em um dos FORMATOS_DATA → DATE; outro formato ou data inexistente → nulo."""
    valor = F.trim(F.col(coluna))
    data = F.lit(None).cast('date')
    for padrao, formato in reversed(FORMATOS_DATA):
        convertida = F.to_date(F.expr(f"try_to_timestamp(trim({coluna}), '{formato}')"))
        data = F.when(valor.rlike(padrao), convertida).otherwise(data)
    return data


mapa_resultado = F.create_map(*[F.lit(x) for par in RESULTADOS.items() for x in par])
df_padronizado = (
    df_bronze.withColumn('id_inspecao', texto('id_inspecao'))
    .withColumn('id_uc', texto('id_uc'))
    .withColumn('inspetor', texto('inspetor'))
    .withColumn('resultado_origem', texto('resultado'))
    .withColumn(
        'resultado',
        F.coalesce(mapa_resultado[F.col('resultado_origem')], F.col('resultado_origem')),
    )
    .withColumn(
        'tipo_irregularidade',
        F.when(
            texto('tipo_irregularidade').isin(MARCADORES_AUSENCIA), F.lit(None).cast('string')
        ).otherwise(texto('tipo_irregularidade')),
    )
    .withColumn('data_origem', F.trim(F.col('data')))
    .withColumn('data', para_data('data'))
    .withColumn(
        'kwh_recuperado',
        F.expr("try_cast(replace(trim(kwh_recuperado), ',', '.') AS DECIMAL(18,2))"),
    )
)

ordem_carga = Window.partitionBy(*COLUNAS_BRONZE).orderBy(
    F.col('ingestion_datetime').asc_nulls_last(), 'source_file', 'sheet'
)
df_tratado = (
    df_padronizado.withColumn('_ordem', F.row_number().over(ordem_carga))
    .where('_ordem = 1')
    .drop('_ordem')
)

## 3. Validação antes de gravar
Cada regra conta as linhas que a **violam**, e todas são calculadas numa única passada. O id repetido entra como `count − count distinct` (linhas a mais).
- **crítica:** interrompe o notebook antes do `overwrite`, e a Silver atual não é tocada. Antes de parar, o notebook mostra até 20 linhas que violam cada regra.
- **aviso:** é exibida, mas não bloqueia a carga.

A coerência entre as colunas é a regra de negócio da base: inspeção confirmada tem tipo de irregularidade; inspeção sem irregularidade não tem tipo nem kWh recuperado. A UC de cada inspeção precisa existir na Silver de cadastro (integridade referencial pedida no case).

In [0]:
ucs_cadastro = (
    spark.table(cadastro_table).select('id_uc').distinct().withColumn('_no_cadastro', F.lit(True))
)
df_validacao = df_tratado.join(ucs_cadastro, 'id_uc', 'left')
REGRA_ID_REPETIDO = 'id_inspecao repetido após remover duplicatas exatas (linhas a mais)'
REGRAS = [  # (regra, condição que identifica uma linha que VIOLA a regra, crítica?)
    (
        'id_inspecao fora do padrão IN + 7 dígitos',
        F.col('id_inspecao').isNull() | ~F.col('id_inspecao').rlike(PADRAO_ID_INSPECAO),
        True,
    ),
    (
        'id_uc fora do padrão UC + 6 dígitos',
        F.col('id_uc').isNull() | ~F.col('id_uc').rlike(PADRAO_ID_UC),
        True,
    ),
    ('id_uc fora da Silver de cadastro', F.col('_no_cadastro').isNull(), True),
    (
        'inspetor fora do padrão INSP-001 a INSP-040',
        F.col('inspetor').isNull() | ~F.col('inspetor').rlike(PADRAO_INSPETOR),
        True,
    ),
    ('data vazia, em formato desconhecido ou inexistente', F.col('data').isNull(), True),
    ('data no futuro', F.col('data') > F.lit(HOJE), True),
    ('data antes do início da base', F.col('data') < F.lit(DATA_MINIMA).cast('date'), True),
    (
        'resultado fora do domínio',
        F.col('resultado').isNull() | ~F.col('resultado').isin(CONFIRMADA, SEM_IRREGULARIDADE),
        True,
    ),
    (
        'irregularidade confirmada sem tipo',
        (F.col('resultado') == CONFIRMADA) & F.col('tipo_irregularidade').isNull(),
        True,
    ),
    (
        'sem irregularidade com tipo ou kWh recuperado',
        (F.col('resultado') == SEM_IRREGULARIDADE)
        & (F.col('tipo_irregularidade').isNotNull() | (F.col('kwh_recuperado') != 0)),
        True,
    ),
    (
        'kwh_recuperado vazio, não convertido ou negativo',
        F.col('kwh_recuperado').isNull() | (F.col('kwh_recuperado') < 0),
        True,
    ),
    (
        'irregularidade confirmada com kwh_recuperado zero',
        (F.col('resultado') == CONFIRMADA) & (F.col('kwh_recuperado') == 0),
        False,
    ),
    (
        'tipo_irregularidade fora dos 5 tipos conhecidos',
        F.col('tipo_irregularidade').isNotNull()
        & ~F.col('tipo_irregularidade').isin(TIPOS_IRREGULARIDADE),
        False,
    ),
]
contagens = df_validacao.agg(
    F.count('*').alias('linhas'),
    (F.count('id_inspecao') - F.count_distinct('id_inspecao')).alias('ids_repetidos'),
    F.count_if(F.col('resultado') == CONFIRMADA).alias('confirmadas'),
    F.count_if(F.col('resultado') == SEM_IRREGULARIDADE).alias('sem_irregularidade'),
    F.sum('kwh_recuperado').alias('kwh_total'),
    F.count_distinct('inspetor').alias('inspetores'),
    F.min('data').alias('primeira_data'),
    F.max('data').alias('ultima_data'),
    *[F.count_if(condicao).alias(f'regra_{i}') for i, (_, condicao, _) in enumerate(REGRAS)],
).first()
linhas_silver = contagens['linhas']
resultado_regras = [(REGRA_ID_REPETIDO, int(contagens['ids_repetidos']), 'crítica')] + [
    (regra, int(contagens[f'regra_{i}']), 'crítica' if critica else 'aviso')
    for i, (regra, _, critica) in enumerate(REGRAS)
]
display(
    spark.createDataFrame(
        [
            (r, v, n, 'OK' if v == 0 else ('FALHA' if n == 'crítica' else 'ATENÇÃO'))
            for r, v, n in resultado_regras
        ],
        'regra string, violacoes long, nivel string, status string',
    )
)
print(
    f'Bronze: {linhas_bronze} linhas | Silver: {linhas_silver} linhas | '
    f'duplicatas removidas: {linhas_bronze - linhas_silver}'
)
print(
    f'Confirmadas: {contagens["confirmadas"]} | sem irregularidade: '
    f'{contagens["sem_irregularidade"]} | energia recuperada: {contagens["kwh_total"]} kWh | '
    f'inspetores: {contagens["inspetores"]} | datas: {contagens["primeira_data"]} a '
    f'{contagens["ultima_data"]}'
)
# Exemplos das regras críticas que falharam (só são calculados nesse caso)
ids_repetidos = df_validacao.groupBy('id_inspecao').count().where('count > 1').drop('count')
exemplos = {REGRA_ID_REPETIDO: df_validacao.join(ids_repetidos, 'id_inspecao')}
exemplos.update({regra: df_validacao.where(condicao) for regra, condicao, _ in REGRAS})
falhas = [f'{r}: {v} linha(s)' for r, v, n in resultado_regras if v and n == 'crítica']
for regra, violacoes, nivel in resultado_regras:
    if nivel == 'crítica' and violacoes:
        print(f'Exemplos de linhas que violam: {regra}')
        display(exemplos[regra].select(*COLUNAS_BRONZE, 'data_origem', 'sheet').limit(20))
if falhas:
    raise RuntimeError('Silver de inspeções reprovada na validação:\n- ' + '\n- '.join(falhas))

regra,violacoes,nivel,status
id_inspecao repetido após remover duplicatas exatas (linhas a mais),0,crítica,OK
id_inspecao fora do padrão IN + 7 dígitos,0,crítica,OK
id_uc fora do padrão UC + 6 dígitos,0,crítica,OK
id_uc fora da Silver de cadastro,0,crítica,OK
inspetor fora do padrão INSP-001 a INSP-040,0,crítica,OK
"data vazia, em formato desconhecido ou inexistente",0,crítica,OK
data no futuro,0,crítica,OK
data antes do início da base,0,crítica,OK
resultado fora do domínio,0,crítica,OK
irregularidade confirmada sem tipo,0,crítica,OK


Bronze: 1389 linhas | Silver: 1389 linhas | duplicatas removidas: 0
Confirmadas: 871 | sem irregularidade: 518 | energia recuperada: 1473112.00 kWh | inspetores: 40 | datas: 2025-08-01 a 2025-12-31


## 4. Escrita na Silver
Só grava com `confirmar_publicacao = true`; com `false`, o notebook termina aqui com a validação feita. Os comentários de coluna vão no schema (`withMetadata`) e são gravados no mesmo commit da carga.

Antes de gravar, o notebook guarda a versão atual da Silver. Depois, confere se a tabela gravada tem as colunas esperadas e o número de linhas validado. Se não tiver, a Silver volta para essa versão (`RESTORE`), ou é removida se esta foi a primeira carga, e o notebook para. O comentário da tabela só é reescrito depois dessa conferência e quando muda; o texto vai escapado (`literal_sql`).

In [0]:
COMENTARIOS = {
    'id_inspecao': 'Identificador da inspeção (IN + 7 dígitos), chave da tabela.',
    'id_uc': 'Unidade consumidora inspecionada; existe na Silver de cadastro.',
    'data': 'Data da inspeção, convertida pelo formato de cada valor (nunca pela aba).',
    'inspetor': 'Código do inspetor (INSP-001 a INSP-040), sem espaços.',
    'resultado': 'IRREGULARIDADE CONFIRMADA ou SEM IRREGULARIDADE.',
    'tipo_irregularidade': 'Tipo da irregularidade confirmada; nulo quando não houve.',
    'kwh_recuperado': 'Energia recuperada apurada na inspeção, em kWh; zero sem irregularidade.',
    'sheet': 'Aba de origem em inspecoes.xlsx (só linhagem: a aba dez tem datas de outros meses).',
    'source_file': 'Arquivo de origem na landing.',
    'ingestion_datetime': 'Momento da carga na Bronze.',
    'silver_datetime': 'Momento da carga na Silver.',
}
COLUNAS_SILVER = list(COMENTARIOS)
COMENTARIO_TABELA = (
    'Silver: inspeções de campo tipadas e padronizadas, uma linha por inspeção. '
    'Regras em docs/silver/silver_inspecoes.md.'
)


def literal_sql(texto):
    """Texto entre aspas simples para SQL, com barra invertida e apóstrofo escapados."""
    return "'" + texto.replace('\\', '\\\\').replace("'", "\\'") + "'"


df_final = df_tratado.withColumn('silver_datetime', F.current_timestamp()).select(*COLUNAS_SILVER)
for coluna, comentario in COMENTARIOS.items():
    df_final = df_final.withMetadata(coluna, {'comment': comentario})

if CONFIRMAR_PUBLICACAO:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.{silver_schema}')
    # Versão atual da Silver, para voltar a ela se a conferência depois da escrita falhar
    versao_anterior = (
        spark.sql(f'DESCRIBE HISTORY {silver_table} LIMIT 1').first()['version']
        if spark.catalog.tableExists(silver_table)
        else None
    )
    (
        df_final.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(silver_table)
    )

    # Qualquer falha daqui em diante (conferência, contagem ou comentário) desfaz a carga
    try:
        tabela = spark.table(silver_table)
        colunas_tabela, linhas_tabela = tabela.columns, tabela.count()
        if colunas_tabela != COLUNAS_SILVER or linhas_tabela != linhas_silver:
            raise RuntimeError(
                f'Silver gravada com {linhas_tabela} linhas e colunas {colunas_tabela}; '
                f'esperado {linhas_silver} linhas e {COLUNAS_SILVER}.'
            )
        if spark.sql(f'DESCRIBE DETAIL {silver_table}').first()['description'] != COMENTARIO_TABELA:
            spark.sql(f'COMMENT ON TABLE {silver_table} IS {literal_sql(COMENTARIO_TABELA)}')
    except Exception as erro:
        try:
            if versao_anterior is None:
                spark.sql(f'DROP TABLE IF EXISTS {silver_table}')
            else:
                spark.sql(f'RESTORE TABLE {silver_table} TO VERSION AS OF {versao_anterior}')
        except Exception as falha:
            raise RuntimeError(
                f'A conferência depois da escrita falhou ({erro}), e a recuperação também: {falha}'
            ) from erro
        raise RuntimeError(
            f'{erro} A carga foi desfeita (versão anterior: {versao_anterior}).'
        ) from erro
    print(f'OK: {linhas_tabela} inspeções carregadas em {silver_table}')
else:
    print(f'Validação concluída; nada gravado em {silver_table} (confirmar_publicacao = false).')

OK: 1389 inspeções carregadas em `v-energy`.silver.tb_inspecoes


## Evidências: antes × depois
As consultas abaixo comparam a Bronze lida no passo 1 (mesma versão) com o resultado da padronização. Elas não alteram dados e rodam também com `confirmar_publicacao = false`.

In [0]:
df_padronizado.createOrReplaceTempView('inspecoes_padronizadas')

# 1. resultado: valores da origem → Silver
display(
    spark.sql("""
  SELECT resultado_origem, resultado, COUNT(*) AS inspecoes
  FROM inspecoes_padronizadas GROUP BY 1, 2 ORDER BY 2, 3 DESC""")
)

# 2. data: formato da origem, aba e mês convertido (a aba dez traz meses anteriores),
# com os mesmos padrões da conversão (FORMATOS_DATA)
formato_origem = F.lit('desconhecido')
for padrao, formato in reversed(FORMATOS_DATA):
    formato_origem = F.when(F.col('data_origem').rlike(padrao), F.lit(formato)).otherwise(
        formato_origem
    )
display(
    df_padronizado.groupBy(
        formato_origem.alias('formato_origem'),
        'sheet',
        F.date_format('data', 'yyyy-MM').alias('mes'),
    )
    .agg(F.count('*').alias('inspecoes'))
    .orderBy('sheet', 'mes', 'formato_origem')
)

# 3. resultado × tipo × kWh: a coerência que a validação exige
display(
    spark.sql("""
  SELECT resultado, coalesce(tipo_irregularidade, '<nulo>') AS tipo_irregularidade,
         COUNT(*) AS inspecoes, SUM(kwh_recuperado) AS kwh_recuperado
  FROM inspecoes_padronizadas GROUP BY 1, 2 ORDER BY 1, 3 DESC""")
)

# 4. Problemas da origem (tabela de problemas de docs/overview_dados/inspecoes.md), na Bronze lida
display(
    df_bronze.agg(
        F.count_if(F.col('inspetor') != F.trim('inspetor')).alias('inspetor_com_espaco'),
        F.count_if(F.col('kwh_recuperado').contains(',')).alias('kwh_com_virgula'),
        F.count_if(~F.trim('data').rlike(FORMATOS_DATA[0][0])).alias('data_fora_do_iso'),
        F.count_if(F.col('tipo_irregularidade').isNull()).alias('tipo_vazio'),
        F.count_if(texto('tipo_irregularidade').isin(MARCADORES_AUSENCIA[1:])).alias(
            'tipo_com_marcador'
        ),
    )
)

resultado_origem,resultado,inspecoes
IRREGULARIDADE CONFIRMADA,IRREGULARIDADE CONFIRMADA,836
CONFIRMADA,IRREGULARIDADE CONFIRMADA,35
SEM IRREGULARIDADE,SEM IRREGULARIDADE,496
NADA CONSTA,SEM IRREGULARIDADE,22


formato_origem,sheet,mes,inspecoes
yyyy-MM-dd,ago-set,2025-08,243
yyyy-MM-dd,ago-set,2025-09,268
MM-dd-yyyy,dez,2025-08,11
dd.MM.yyyy,dez,2025-08,8
dd/MM/yyyy,dez,2025-08,5
MM-dd-yyyy,dez,2025-09,7
dd.MM.yyyy,dez,2025-09,6
dd/MM/yyyy,dez,2025-09,10
MM-dd-yyyy,dez,2025-10,15
dd.MM.yyyy,dez,2025-10,10


resultado,tipo_irregularidade,inspecoes,kwh_recuperado
IRREGULARIDADE CONFIRMADA,DESVIO DIRETO,182,296200.20
IRREGULARIDADE CONFIRMADA,BY-PASS,181,293882.80
IRREGULARIDADE CONFIRMADA,MEDIDOR ADULTERADO,180,299408.90
IRREGULARIDADE CONFIRMADA,LACRE VIOLADO,164,294205.90
IRREGULARIDADE CONFIRMADA,LIGACAO CLANDESTINA,164,289414.20
SEM IRREGULARIDADE,,518,0.00


inspetor_com_espaco,kwh_com_virgula,data_fora_do_iso,tipo_vazio,tipo_com_marcador
83,125,133,311,207
